<a href="https://colab.research.google.com/github/gisalgs/notebooks/blob/main/point-in-polygon-4-colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Point-in-polygon: the even-odd algorithm

Point-in-polygon is a crucial operation when we handle spatial data. For example, if we click on a polygon in QGIS, we want to quickly pull out the data about this polygon (instead of waiting for a while before the data shows up). There are many ways to determine whether a point is in a polygon and we introduce the **even-odd algorithm** here. To determine if a point (such as one of the dots in following figure) is in a polygon, we first draw a half line that starts from the point and points to the right of the point. If the half line intersects with the polygon in an odd number of times, then we say the point is in the polygon. Otherwise the point is outside the polygon. 

<img width=350px src="figures/pip-1.png"/>

This seemingly simple algorithm, however, has many special cases that are tedious and must be addressed. For example, the half line *a* in the figure has a part of the polygon boundary on it and we will have to decide how many times the half lines intersects the polygon. Also, when a half line (e.g., *b* and *c*) goes through a vertex of the polygon, how many times should we count the intersection?

The next consideration is the calculation of intersection points. Computing the physical intersection point, if any, is an expensive process -- it takes some time to do and it takes a lot of times if we have a lot of intersection points to compute. To understand this, we should remember the so called polygon in GIS has many line segments and we will have to test whether each of the line segments intersects with the half line. That sounds a lot. But the good news is: we don't have to get the physical intersection point for most of the cases. For many cases, we just need to make a quick decision whether the half line intersects with the boundary. We can illustrate this using the following figure. 

<img width=350px src="figures/pip-2.png"/>

In the above figure, it is clear that the half line (the one starts at point *A*) is not going to intersect with line segments *a*, *d*, and *e*. Line segment *c* crosses the half line, but we don't need to compute the intersection point --- we know for fact they will intersect because the two end points of *c* are on the different sides of the half line, and are to the right of point *A*. Taking out these cases, we only have one case left: line segments like *b* or *b'* where the two end points are on both sizes of the half line but one of the end point is to the left of point *A*. 

Then we need to consider a few **special cases**, when the point (dark dot below) is on one of the two ends of the line segment, or on a horizontal or vertical line.

<img width=450px src="figures/point-in-polygon-special-cases.png"/>

In this tutorial, we are not going to discuss the details of calculating intersection of two line segments. Please refer to sections 2.5 and 2.6 in the *GIS Algorithms* book for those details. 



## Polygon Representation

We represent a polygon of $n$ points as $(x_1, y_1), (x_2, y_2),...,(x_n,y_n),(x_{n+1},y_{n+1})$, where we add an additional point in the sequence $(x_{n+1}, y_{n+1}) = (x_1,y_1)$ to ensure that the polygon is closed, a similar requirement as we saw in GeoJSON.

But what if a polygon is not closed?

## Handling exceptions

One important issue in the above algorithm is that we require the polygon to be closed. Otherwise, there will be errors. There are different ways to handle this. For example, we can simply return a `None` if the polygon is not closed. However, to make the code look more formal, we are going to tell Python that there is an error (in this way, we don't let errors go without being notified) and we want to stop running the code (because something important might just be dependent on the successful calculation of the area). This is what we call to **raise an exception** or error. We will simply use a generic exception in Python to do this. In the following code, we will see the program stops after it prints 5 and then trows an exception:

In [ ]:
def dummy():
    for i in range(10):
        print(i, end=' ')
        if i == 5:
            raise Exception('Just an error')

dummy()        
print('DONE')


It appears that we have successfully raised an error (exception) because the program is stopped because the above program never gets to the last line. But should we allow the program to continue? This is especially a legitimate question if we know how to deal with the error. We can actually catch the exception and process it so that we decide if we need to end the program or not. This is to use the `try...except` statement:

In [ ]:
# TODO: Complete this code block to show how try...except statement works
#




## Implementing the even-odd algorithm

The even-odd algorithm for determining if a point is in a polygon is implemented as function `pip_cross` in [point_in_polygon.py](https://github.com/gisalgs/geom/blob/master/point_in_polygon.py). It is listed below. This code was adopted from the C program in Graphics Gems IV by Haines (1994). 

```python
 1. def pip_cross(point, pgon):
 2.     if pgon[0] != pgon[-1]:
 3.         raise Exception('Polygon not closed')
 4.     x, y = point.x, point.y
 5.     N = len(pgon)
 6.     crossing_count = 0
 7.     is_point_inside = False
 8.     for i in range(N-1):
 9.         p1, p2 = pgon[i], pgon[i+1]
10.         yside1 = p1.y >= y
11.         yside2 = p2.y >= y
12.         xside1 = p1.x >= x
13.         xside2 = p2.x >= x
14.         if (p1.y == p2.y == y and (xside1 != xside2 or x == p1.x or x == p2.x)) or \
15.         (p1.x == p2.x == x and (yside1 != yside2 or y == p1.y or y == p2.y)) or \
16.         p1 == point or \
17.         p2 == point:
18.             crossing_count = 1
19.             is_point_inside = True
20.             return is_point_inside, crossing_count
21.         if yside1 != yside2:
22.             if xside1 == xside2:
23.                 if xside1:
24.                     crossing_count += 1
25.                     is_point_inside = not is_point_inside
26.             else:
27.                 m = p2.x - (p2.y-y)*(p1.x-p2.x)/(p1.y-p2.y)
28.                 if m == x:
29.                     crossing_count = 1
30.                     is_point_inside = True
31.                     return is_point_inside, crossing_count
32.                 elif m > x:
33.                     crossing_count += 1
34.                     is_point_inside = not is_point_inside
35.     return is_point_inside, crossing_count
```


To fully understand the code, we should try some of the half-lines and see whether the function returns the right result. We also assume the right-most point is the first point of the polygon and the sequence of points goes counterclockwise. The order the line segments are marked in red in the following figure.

<img width=450px src="figures/pip-1x.png"/>

Let's start with half-line **a**. We are going to check the algorithm for each line segment. For every segment, we get the values of `yside1`, `yside2`, `xside1`, and `xside2`. To check line segment <font color="red">1</font> against half-line a, for example, we have the following:

```
yside1 = True, because p1.y >= y  
yside2 = True, because p2.y >= y  
xside1 = True, because p1.x >= x  
xside2 = True, because p2.x >= x
```

where p1 and p2 are the begin and end points of the line segment, respectively, and x and y are the coordinates of starting point of the half-line.

Below, the first number in each line refers to the line segment as marked on the figure. When we refer to the line numbers of the `pip_cross` function, the numbers are shown in the above code listing.

| Line# | Conditions | Analysis |
|:-----:|---------------|-------------|
| <font color="red">1</font> | yside1 = True, yside2 = True | Conditions in lines 14-17, 21 are False. Next  |
| <font color="red">1</font> | yside1 = True, yside2 = True | Conditions in lines 14-17, 21 are False. Next  |
| <font color="red">2</font> | yside1 = True, yside2 = True | ut supra (also check >= in lines 10-13)  |
| <font color="red">3</font> | yside1 = True, yside2 = True | idem  |
| <font color="red">4</font> | yside1 = True, yside2 = True | idem  |
| <font color="red">5</font> | yside1 = True, yside2 = False, xside1 = True, xside2 = True | Conditions in lines 21, 22, 23 are True<br/>cross_count = 1, is_point_inside = True |
| <font color="red">6</font> | ut supra | cross_count = 2, is_point_inside = False  |
| <font color="red">7</font> | idem | cross_count = 3, is_point_inside = True    |
| <font color="red">8</font> | idem | cross_count = 4, is_point_inside = False  |
| <font color="red">9</font> | idem | cross_count = 5, is_point_inside = True    |
| <font color="red">10</font> | idem | cross_count = 6, is_point_inside = False  |
| <font color="red">11</font> | idem | cross_count = 7, is_point_inside = True    |
| <font color="red">12</font> | idem | cross_count = 8, is_point_inside = False |

(Note: ut supra - as stated above or as above, idem - same)

So the final result is `(False, 8)`, meaning there are 8 intersections and the point forming the half line a is not in the polygon.

If we consider half-line **b**, here is what we have:

| Line# | Conditions | Analysis |
|:-----:|---------------|-------------|
| <font color="red">1</font> | yside1 = yside2 = True | Conditions in lines 14-17, 21 are False |
| <font color="red">2</font> | yside1 = True, yside2 = False, xside1 = xside2 = False | Lines 14-17 False, lines 21-22 True, line 23 False |
| <font color="red">3</font> | yside1 = yside2 = False, xside1 = xside2 = False | Lines 14-17, 21 False |
| <font color="red">4</font> | yside1 = False, yside2 = True, xside1 = xside2 = False | Lines 14-17, False, lines 21-22 True, line 23 False |
| <font color="red">5</font> | yside1 = True, yside2 = False, xside1 = xside2 = False | ut supra  |
| <font color="red">6</font> | yside1 = False, yside2 = True, xside1 = xside2 = False | idem |
| <font color="red">7</font> | yside1 = True, yside2 = False, xside1 = xside2 = False | idem |
| <font color="red">8</font> | yside1 = False, yside2 = True, xside1 = False, xside2 = True |Conditions in lines 14-17 False, line 21 True, line 22 False, line 26 True. Calculate m and we have m > x because intersection is on the right side of the point. line 32 is True<br/>cross_count = 1, is_point_inside = True  |
| <font color="red">9</font> | yside1 = True, yside2 = False, xside1 = xside2 = True | Conditions in lines 14-17 False, line 21-23 True<br/>cross_count = 2, is_point_inside = False  |
| <font color="red">10</font> | yside1 = False, yside2 = True, xside1 = xside2 = True | ut supra<br/>cross_count = 3, is_point_inside = True  |
| <font color="red">11</font> | yside1 = True, yside2 = False, xside1 = xside2 = True | idem<br/>cross_count = 4, is_point_inside = False  |
| <font color="red">12</font> | yside1 = False, yside2 = True, xside1 = xside2 = True | idem<br/>cross_count = 5, is_point_inside = True  |

This time it is a little more subtle because the half-line "crosses" 5 line segments even though it only touches one of the end points, and some of the end points are double counted. So we should know that the count may not make sense, but the detection of whether the point is in or not should always be right. Of course we decide that a point on the boundary will be considered as in the polygon, which may not be desired in some cases.

### Using Jupyter Notebook and Colab

The complete version of the above code is available from the github site at [here](https://github.com/gisalgs/geom/blob/master/point_in_polygon.py), where the code is saved in a file called **point_in_polygon.py**. We can put it in the same folder of geom as the point.py file. (Note that the above code is simply listed here but not executed.) Now we can use the centroid algorithm in the following example. 

This tutorial will be run on-the-fly in Google Colab. Because of that, it is necessary to load our geometry Python modules (such as point.py). The following line of code clones exactly the entire repository into the running environment of his notebook. Please note this is only needed to run in colab. To run the code in the tutorial in our own local computer using a Python interactive mode, we will use what we did before (i.e., downloading the necessary .py files from the github repo, and putting them in the right folder). There is no need to run this in local Python environment. This line should not be included in a Python program either.

In [ ]:
!git clone https://github.com/gisalgs/geom.git 

The URL `https://github.com/gisalgs/geom.git` can be found from the repo at [https://github.com/gisalgs/geom](https://github.com/gisalgs/geom) (clicking on the green Code button).

The above line may show an error that the path exists already. This happens when it is run multiple times. No need to alert. If we are curious about where the module is actually installed, we can use the following Linux commands to show some information of the folders and what we have cloned.

In [ ]:
!ls ./geom

In [ ]:
!pwd

We can also do this in Python:

In [ ]:
import os
os.getcwd()

Once the github repo is cloned, we can import necessary modules and test the algorithm using some artificial data below. 

In [ ]:
from geom.point import *
from geom.point_in_polygon import *

points1 = [ [0,10], [5,0], [10,10], [15,0], [20,10],
           [25,0], [30,20], [40,20], [45,0], [60, 50], [50,50],
           [40,40], [30,50], [25,20], [20,50], [15,10],
           [10,50], [8, 8], [4,50], [0,10] ]
polygon1 = [Point(p[0], p[1]) for p in points1]

p1 = Point(10, 30)
p2 = Point(10, 20)
p3 = Point(20, 40)
p4 = Point(5, 40)
p5 = Point(35, 20)
p6 = Point(55, 50)

words = ['OUT', 'IN']

for p in [p1, p2, p3, p4, p5, p6]:
    print(f'Point {p} is {words[pip_cross(p, polygon1)[0]]}') # 

In the above, we are treating a boolean value as an integer (`True` as 1 and `False` as 0). This seems to be wild. But consider this:

In [ ]:
True+True, True+False, True-True, 6*True

Below we will show a visual proof of the above results. Before we plot the figure, we need to import matplotlib first:

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

We also need a little function as a helper to translate True and False values to 1 and 0, respectively.

We will use the above little function to automatically assign two colors: if the point is in then we make it blue, otherwise red (see the line starts with `colors` below.

In [ ]:
# ax = plt.gca()
_, ax = plt.subplots()

line1 = polygon1 # [ [p.x, p.y] for p in polygon1 ]
pg1 = plt.Polygon(line1, closed=True, fill=True, 
                  facecolor='#AAAAAA', edgecolor='grey', alpha=0.5)
ax.add_patch(pg1)

pts = [p1, p2, p3, p4, p5, p6]

results = [pip_cross(p, polygon1)[0] for p in pts] # do this without using in_or_out?

colors = [['red', 'blue'][i] for i in results]

l2 = plt.scatter([p.x for p in pts], [p.y for p in pts], color=colors, s=15)
labels = ['p%s'%(i+1) for i in range(len(pts))]
for i, p in enumerate(pts):
    plt.text(p.x+1, p.y+1, labels[i], color=colors[i])
  
ax.set_aspect(1)
plt.show()

Note the above code assumes the polygon is properly closed. If we cannot guarantee, the actual code will be more tedious because we will have to use the try statement:

In [ ]:
points2 = [ [0,10], [5,0], [10,10], [15,0], [20,10] ]
polygon2 = [ [ Point(p[0], p[1]) for p in points2 ] ]

try:
    print(f'Point {p1} is {words[pip_cross2(p1, polygon2)[0]]}')
except Exception as err:
    print(err)

## Case study: Polygon area using Monte Carlo simulation

We are going to use what we have learned so far to do an experiment. 

First, let's make a triangle that has a minimal bounding box (envelope) defined as `[0, 10, 0, 10]`, which are the lower bound and upper bound of x, and that of y, respectively. 

In [ ]:
# TODO: create a triangle here. 
#       It will be a list of four Point objects, 
#       with the first and last being the same point.
#       Make sure its *minimum bounding box is [0, 10, 0, 10]
#

triangle = 

Make sure we can draw the triangle, along with its bounding box.

In [ ]:
# TODO: Draw triangle and bounding box here
#



Now we randomly generate 100 points.

In [ ]:
# TODO: generate 100 random points with the bounding box
import random

points = 


Count how many points are within the triangle.

In [ ]:
# TODO: count the number of points in the triangle
#
# It will be OK to use the sum() function in Python.




Calculate the size of the triangle. Use the proportion of the points within the triangle and the size of the bounding box.

In [ ]:
# TODO: compute the size of the triangle




In [ ]:
# TODO: draw the triangle, bounding box, and points
#       points in the triangle are rendered blue
#       points outside are red



At this point, we should get a value that is supposed to be the size of the triangle. But this is just one estimation. If we do this again, we would get a different answer. If we repeat this process 100 times, each time we have an estimate of the size of the triangle and we can plot a histogram that should tell us a lot about the size of the triangle. We can calculate the mean of all the estimated sizes. We can also draw the points with the triangle to get some kind of visual proof. This experiment can go on (for example, we can use 1000 points instead of 100). This is generally what we mean by a Monte Carlo simulation. 

In [ ]:
# TODO: use a loop of 100 iterations to repeat the above process of 
#       generating 100 random points and estimate the triangle size. 
#       In every iteration, we will an estimate of the area.
#       We put all the estimate in a list called areas
#
#       It will be a good idea to write a function to estimate the size
#       of the triangle using the number of points in the triangle.



When all is done (100 iterations and 100 areas), the list `areas` will have 100 values and we draw a histogram using the following code.

In [ ]:
_, ax = plt.subplots()
ax.hist(areas, 10, facecolor='grey', edgecolor='black', alpha=0.75)
plt.show()